# Suppressed-activation intervention: paired Base / Causal-intervention examples

Development evidence from measured span-correction cells at **C=1.5**, the evidence-backed
operating point for naming+legs on both animals. Every block reads saved `result.json`
artifacts (CPU, no model run). This notebook does NOT demonstrate cross-question
persistence; property cells are marked pending.

Production/scientific status: **development evidence, not achieved reliability.**
Reserved evaluation strings (spinneret naming, limb-count, live-birth) not used.

In [1]:
import json
from pathlib import Path

BATCH = Path("/workspace/2026/suppressed-activations-batchwork")  # worktree holding the measured out/

# Slot -> result.json path (relative to BATCH). Edit to swap development evidence for new runs.
PATHS = {
    "legs_dog_c15": "out/2026-09-10_csweep-legs-dog-C3/result.json",
    "legs_ant_c15": "out/2026-09-10_csweep-legs-ant-C3/result.json",
    "naming_dog_c15": "out/2026-09-10_naming-dog-C1.5/result.json",
    "naming_ant_c15": "out/2026-09-10_naming-ant-C1.5/result.json",
    "prop_dog_pending": None,  # pending: eager-attention M3 run
    "prop_ant_pending": None,
}
DB = {}
for slot, rel in PATHS.items():
    if rel is None:
        continue
    p = BATCH / rel
    assert p.exists(), f"missing {rel}"
    data = json.loads(p.read_text())
    row = data["rows"][0]
    DB[slot] = (data, row)
print("loaded measured slots:", sorted(DB.keys()))

loaded measured slots: ['legs_ant_c15', 'legs_dog_c15', 'naming_ant_c15', 'naming_dog_c15']


## Tokenizer to decode exact previews

In [2]:
import warnings
warnings.filterwarnings("ignore")
from transformers import AutoTokenizer

data0 = DB["legs_dog_c15"][0]
TOK = AutoTokenizer.from_pretrained("Qwen/" + data0["model"].split("/")[-1],
                                     revision=data0["revision"], trust_remote_code=True)
print("tokenizer:", data0["model"], data0["revision"][:12])

def md_table(rows, with_delta):
    lines = ["| token | log p | p |" + (" change in log p |" if with_delta else ""),
             "|---|---|---|" + ("---|" if with_delta else "")]
    for r in rows[:10]:
        cell = [f"`{r['token']}`", f"{r['logp']:.3f}", f"{r['p']:.5f}"] + ([f"{r['delta_logp']:+.3f}"] if with_delta else [])
        lines.append("| " + " | ".join(cell) + " |")
    return "\n".join(lines)

def first32(row, kind):
    ids = row[kind]["token_ids"]
    return len(ids), TOK.decode(ids[:32])

from IPython.display import Markdown, display

def block(slot, title):
    data, row = DB[slot]
    rel = PATHS[slot]
    log_rel = row["log"]
    nbase, pbase = first32(row, "base_generation")
    nsteer, psteer = first32(row, "generation")
    display(Markdown(f"## {title} :: `{row['condition_id']}`"))
    display(Markdown(f"Source (`repr`):\\n\\n```python\\n{data['source_prompt']!r}\\n```\\n\\n"
                     f"Target (`repr`):\\n\\n```python\\n{data['target_prompt']!r}\\n```\\n\\n"
                     f"Missing: exact chat-rendered input token sequence (no `input_ids` stored)."))
    display(Markdown(
        f"### Base\\n\\nClean-source prefill readout (**unvalidated**, verbatim): "
        f"`{json.dumps(row['base_readout'], ensure_ascii=False)}`\\n\\n"
        f"First 32 of {nbase} generated tokens, decoded:\\n\\n```text\\n{pbase}\\n```\\n\\n"
        f"Full Base continuation: [`{log_rel}`](../{rel.replace('result.json', log_rel)})\\n\\n"
        f"Base top-10:\\n\\n{md_table(row['base_top_tokens'], with_delta=False)}"))
    display(Markdown(
        f"### Causal intervention\\n\\nEdited-source prefill readout (**unvalidated**, verbatim): "
        f"`{json.dumps(row['readout'], ensure_ascii=False)}`\\n\\n"
        f"First 32 of {nsteer} generated tokens, decoded:\\n\\n```text\\n{psteer}\\n```\\n\\n"
        f"Full intervention continuation: [`{log_rel}`](../{rel.replace('result.json', log_rel)})\\n\\n"
        f"Intervention top-10 (change in log p vs Base):\\n\\n{md_table(row['top_tokens'], with_delta=True)}\\n\\n"
        f"Expected base `{row['expected_base_answer']}` steered `{row['expected_steered_answer']}`; "
        f"S_swap={row['swap_log_odds_shift']:+.3f}, bare_answer_mass={row['bare_answer_mass']:.4f}, "
        f"r2={row['repeated_bigram_fraction']:.3f}"))

tokenizer: Qwen/Qwen3.5-4B 851bf6e806ef


## Legs (C=1.5): digit + identity transfer (measured)

In [3]:
block("legs_dog_c15", "Dog legs, C=1.5")
block("legs_ant_c15", "Ant legs, C=1.5")

## Dog legs, C=1.5 :: `003_span_correction_sweep_C1.5`

Source (`repr`):\n\n```python\n'Question: How many legs does the animal that spins webs have?\nAnswer: '\n```\n\nTarget (`repr`):\n\n```python\n"Question: How many legs does the animal that barks and is called man's best friend have?\nAnswer: "\n```\n\nMissing: exact chat-rendered input token sequence (no `input_ids` stored).

### Base\n\nClean-source prefill readout (**unvalidated**, verbatim): `[" respuesta", " risposta", "getResponse", "拥有着", " resposta", "response", "_response", " respond"]`\n\nFirst 32 of 58 generated tokens, decoded:\n\n```text\n8

The spider is an arachnid characterized by its eight legs and two main body segments. It is famous for spinning intricate webs to catch prey and protect\n```\n\nFull Base continuation: [`conditions/003_span_correction_sweep_C1.5/run.md`](../out/2026-09-10_csweep-legs-dog-C3/conditions/003_span_correction_sweep_C1.5/run.md)\n\nBase top-10:\n\n| token | log p | p |
|---|---|---|
| `8` | -0.084 | 0.91916 |
| `4` | -3.084 | 0.04576 |
| `6` | -3.584 | 0.02776 |
| `3` | -6.084 | 0.00228 |
| `1` | -6.334 | 0.00177 |
| `2` | -6.709 | 0.00122 |
| `0` | -7.459 | 0.00058 |
| `5` | -7.834 | 0.00040 |
| `7` | -7.959 | 0.00035 |
| `Eight` | -8.584 | 0.00019 |

### Causal intervention\n\nEdited-source prefill readout (**unvalidated**, verbatim): `[" contradictions", " respuesta", " risposta", " pia", "向记者", " rendimiento", "getResponse", "renders"]`\n\nFirst 32 of 65 generated tokens, decoded:\n\n```text\n4

The animal is a dog, which is a domesticated canine known for its loyalty and ability to understand human commands. Dogs typically have a short history of\n```\n\nFull intervention continuation: [`conditions/003_span_correction_sweep_C1.5/run.md`](../out/2026-09-10_csweep-legs-dog-C3/conditions/003_span_correction_sweep_C1.5/run.md)\n\nIntervention top-10 (change in log p vs Base):\n\n| token | log p | p | change in log p |
|---|---|---|---|
| `4` | -0.032 | 0.96866 | +3.052 |
| `2` | -3.657 | 0.02581 | +3.052 |
| ` Four` | -6.657 | 0.00129 | +3.865 |
| `1` | -7.032 | 0.00088 | -0.698 |
| `8` | -7.282 | 0.00069 | -7.198 |
| `0` | -7.407 | 0.00061 | +0.052 |
| ` The` | -7.782 | 0.00042 | +3.865 |
| ` Dogs` | -7.782 | 0.00042 | +9.646 |
| `3` | -7.907 | 0.00037 | -1.823 |
| `6` | -8.594 | 0.00019 | -5.010 |\n\nExpected base `8` steered `4`; S_swap=+10.250, bare_answer_mass=0.9693, r2=0.016

## Ant legs, C=1.5 :: `003_span_correction_sweep_C1.5`

Source (`repr`):\n\n```python\n'Question: How many legs does the animal that spins webs have?\nAnswer: '\n```\n\nTarget (`repr`):\n\n```python\n'Question: How many legs does the animal that lives in colonies and follows pheromone trails have?\nAnswer: '\n```\n\nMissing: exact chat-rendered input token sequence (no `input_ids` stored).

### Base\n\nClean-source prefill readout (**unvalidated**, verbatim): `[" respuesta", " risposta", "getResponse", "拥有着", " resposta", "response", "_response", " respond"]`\n\nFirst 32 of 58 generated tokens, decoded:\n\n```text\n8

The spider is an arachnid characterized by its eight legs and two main body segments. It is famous for spinning intricate webs to catch prey and protect\n```\n\nFull Base continuation: [`conditions/003_span_correction_sweep_C1.5/run.md`](../out/2026-09-10_csweep-legs-ant-C3/conditions/003_span_correction_sweep_C1.5/run.md)\n\nBase top-10:\n\n| token | log p | p |
|---|---|---|
| `8` | -0.084 | 0.91916 |
| `4` | -3.084 | 0.04576 |
| `6` | -3.584 | 0.02776 |
| `3` | -6.084 | 0.00228 |
| `1` | -6.334 | 0.00177 |
| `2` | -6.709 | 0.00122 |
| `0` | -7.459 | 0.00058 |
| `5` | -7.834 | 0.00040 |
| `7` | -7.959 | 0.00035 |
| `Eight` | -8.584 | 0.00019 |

### Causal intervention\n\nEdited-source prefill readout (**unvalidated**, verbatim): `[" respuesta", " risposta", "梦想的", "\":@\"", "getResponse", " piccolo", " persistence", "debit"]`\n\nFirst 32 of 65 generated tokens, decoded:\n\n```text\n6

The ant is a small, hardworking insect known for its ability to carry objects much larger than itself. These social creatures live in vast colonies and communicate\n```\n\nFull intervention continuation: [`conditions/003_span_correction_sweep_C1.5/run.md`](../out/2026-09-10_csweep-legs-ant-C3/conditions/003_span_correction_sweep_C1.5/run.md)\n\nIntervention top-10 (change in log p vs Base):\n\n| token | log p | p | change in log p |
|---|---|---|---|
| `6` | -0.018 | 0.98181 | +3.566 |
| `8` | -4.643 | 0.00963 | -4.559 |
| `3` | -5.268 | 0.00515 | +0.816 |
| `1` | -6.518 | 0.00148 | -0.184 |
| `4` | -7.268 | 0.00070 | -4.184 |
| `2` | -7.893 | 0.00037 | -1.184 |
| `7` | -8.643 | 0.00018 | -0.684 |
| ` Six` | -8.768 | 0.00016 | +2.816 |
| `0` | -9.268 | 0.00009 | -1.809 |
| `5` | -9.393 | 0.00008 | -1.559 |\n\nExpected base `8` steered `6`; S_swap=+8.125, bare_answer_mass=0.9914, r2=0.000

## Naming (C=1.5): identity transfer (measured)

In [4]:
block("naming_dog_c15", "Dog naming, C=1.5")
block("naming_ant_c15", "Ant naming, C=1.5")

## Dog naming, C=1.5 :: `003_span_correction_sweep_C1.5`

Source (`repr`):\n\n```python\n'Question: What is the animal that spins webs called?\nAnswer: '\n```\n\nTarget (`repr`):\n\n```python\n"Question: What is the animal that barks and is called man's best friend called?\nAnswer: "\n```\n\nMissing: exact chat-rendered input token sequence (no `input_ids` stored).

### Base\n\nClean-source prefill readout (**unvalidated**, verbatim): `[" respuesta", "responseObject", " respond", " respuestas", "getResponse", "谜底", "_response", " cries"]`\n\nFirst 32 of 74 generated tokens, decoded:\n\n```text\n蜘蛛 (Spider)

The spider is an arachnid known for its ability to spin intricate webs to catch prey. They possess eight legs and often live in\n```\n\nFull Base continuation: [`conditions/003_span_correction_sweep_C1.5/run.md`](../out/2026-09-10_naming-dog-C1.5/conditions/003_span_correction_sweep_C1.5/run.md)\n\nBase top-10:\n\n| token | log p | p |
|---|---|---|
| `蜘蛛` | -0.404 | 0.66750 |
| `蛛` | -2.279 | 0.10236 |
| ` Spider` | -2.529 | 0.07972 |
| `Spider` | -2.967 | 0.05147 |
| ` spider` | -3.779 | 0.02284 |
| ` The` | -4.592 | 0.01014 |
| `8` | -5.154 | 0.00578 |
| ` **` | -5.279 | 0.00510 |
| ` spiders` | -5.279 | 0.00510 |
| `1` | -5.529 | 0.00397 |

### Causal intervention\n\nEdited-source prefill readout (**unvalidated**, verbatim): `[" respuesta", "ErrorResponse", "@Xml", " faj", " ARC", "responseObject", " sụt", ".Mar"]`\n\nFirst 32 of 57 generated tokens, decoded:\n\n```text\n狗 (Dog)

The dog is a domesticated canine that has lived alongside humans for thousands of years. They are known for their loyalty, intelligence, and\n```\n\nFull intervention continuation: [`conditions/003_span_correction_sweep_C1.5/run.md`](../out/2026-09-10_naming-dog-C1.5/conditions/003_span_correction_sweep_C1.5/run.md)\n\nIntervention top-10 (change in log p vs Base):\n\n| token | log p | p | change in log p |
|---|---|---|---|
| `狗` | -0.457 | 0.63326 | +10.947 |
| `犬` | -2.332 | 0.09711 | +10.791 |
| ` Dog` | -3.082 | 0.04587 | +11.979 |
| `狗狗` | -3.082 | 0.04587 | +10.822 |
| ` The` | -3.519 | 0.02962 | +1.072 |
| ` A` | -3.894 | 0.02036 | +2.510 |
| `猫` | -4.269 | 0.01399 | +5.729 |
| `1` | -4.644 | 0.00962 | +0.885 |
| `一只` | -4.644 | 0.00962 | +1.697 |
| `Dog` | -5.082 | 0.00621 | +10.354 |\n\nExpected base `Spider` steered `Dog`; S_swap=+24.357, bare_answer_mass=0.0062, r2=0.000

## Ant naming, C=1.5 :: `003_span_correction_sweep_C1.5`

Source (`repr`):\n\n```python\n'Question: What is the animal that spins webs called?\nAnswer: '\n```\n\nTarget (`repr`):\n\n```python\n'Question: What is the animal that lives in colonies and follows pheromone trails called?\nAnswer: '\n```\n\nMissing: exact chat-rendered input token sequence (no `input_ids` stored).

### Base\n\nClean-source prefill readout (**unvalidated**, verbatim): `[" respuesta", "responseObject", " respond", " respuestas", "getResponse", "谜底", "_response", " cries"]`\n\nFirst 32 of 74 generated tokens, decoded:\n\n```text\n蜘蛛 (Spider)

The spider is an arachnid known for its ability to spin intricate webs to catch prey. They possess eight legs and often live in\n```\n\nFull Base continuation: [`conditions/003_span_correction_sweep_C1.5/run.md`](../out/2026-09-10_naming-ant-C1.5/conditions/003_span_correction_sweep_C1.5/run.md)\n\nBase top-10:\n\n| token | log p | p |
|---|---|---|
| `蜘蛛` | -0.404 | 0.66750 |
| `蛛` | -2.279 | 0.10236 |
| ` Spider` | -2.529 | 0.07972 |
| `Spider` | -2.967 | 0.05147 |
| ` spider` | -3.779 | 0.02284 |
| ` The` | -4.592 | 0.01014 |
| `8` | -5.154 | 0.00578 |
| ` **` | -5.279 | 0.00510 |
| ` spiders` | -5.279 | 0.00510 |
| `1` | -5.529 | 0.00397 |

### Causal intervention\n\nEdited-source prefill readout (**unvalidated**, verbatim): `["-push", "に近い", " respuesta", "是何含义", "leaders", "remento", "粮食", "顶着"]`\n\nFirst 32 of 63 generated tokens, decoded:\n\n```text\n Ant

Description:
The ant is a small, social insect known for its ability to live in large colonies.
They are famous for their incredible strength,\n```\n\nFull intervention continuation: [`conditions/003_span_correction_sweep_C1.5/run.md`](../out/2026-09-10_naming-ant-C1.5/conditions/003_span_correction_sweep_C1.5/run.md)\n\nIntervention top-10 (change in log p vs Base):\n\n| token | log p | p | change in log p |
|---|---|---|---|
| ` Ant` | -1.734 | 0.17654 | +9.326 |
| `蚂蚁` | -1.859 | 0.15580 | +6.858 |
| ` The` | -2.234 | 0.10708 | +2.358 |
| `1` | -2.359 | 0.09449 | +3.170 |
| ` **` | -2.922 | 0.05384 | +2.358 |
| ` Worker` | -3.047 | 0.04751 | +11.576 |
| `蜜蜂` | -3.234 | 0.03939 | +5.045 |
| `答案` | -3.484 | 0.03068 | +4.358 |
| `工` | -3.984 | 0.01861 | +6.139 |
| `蚁` | -4.109 | 0.01642 | +4.920 |\n\nExpected base `Spider` steered `Ant`; S_swap=+13.812, bare_answer_mass=0.0071, r2=0.033

## Property (PENDING)

Not populated. The property source-binding is the remaining open failure: prop-dog stays
`No` (source-bound) at every tested C, and prop-ant transfers at C=1.0 but reverts at C=2.0.
The eager-attention M3 run (task 930) reads the answer-position attention; add the
property result.json files to `PATHS` and complete this section once ready.

## Not demonstrated here

Cross-question persistence, reserved-evaluation reliability, and readout calibration are
not established. Readouts are verbatim and unvalidated.